# Replication Results Notebook

## Table 1

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

In [7]:
ACS_AGG_RESULTS_PATH = Path("/scratch/gpfs/MSALGANIK/sarahep/folktexts-replication-results") / "aggregated_results.2026.03.22-16.50.13.csv"
results_df = pd.read_csv(ACS_AGG_RESULTS_PATH, index_col=0)

# Convert string "True"/"False" columns to proper booleans
bool_cols = ["uses_chat_template", "config_numeric_risk_prompting", "is_inst", "config_use_chat_template"]
for col in bool_cols:
    if col in results_df.columns:
        results_df[col] = results_df[col].map({"True": True, "False": False, True: True, False: False})

print(f"{results_df.shape=}")
results_df.head(2)

results_df.shape=(16, 73)


,accuracy,accuracy_diff,accuracy_ratio,balanced_accuracy,balanced_accuracy_diff,balanced_accuracy_ratio,benchmark_hash,brier_score_loss,current_time,ece,...,is_inst,uses_chat_template,num_features,uses_all_features,fit_thresh_on_100,fit_thresh_accuracy,optimal_thresh,optimal_thresh_accuracy,score_stdev,score_mean
Llama-3-8B-Instruct__ACSIncome__-1__Num_Chat,0.737699,0.056741,0.927393,0.757311,0.029265,0.961714,3186524256,0.189771,2026.02.22-17.55.39,0.146669,...,True,True,-1,True,0.454450,0.665149,0.630350,0.746543,0.205012,0.504334
Llama-3-8B-Instruct__ACSIncome__-1__QA_Chat,0.706446,0.059889,0.920509,0.742435,0.021565,0.970811,1486026996,0.206004,2026.02.23-01.43.50,0.189902,...,True,True,-1,True,0.739818,0.758450,0.611241,0.741286,0.349886,0.557779


In [8]:
# Column references
model_col = "config_model_name"
task_col = "config_task_name"
numeric_prompt_col = "config_numeric_risk_prompting"

# Metrics to include in the table
table_metrics = ["ece", "brier_score_loss", "roc_auc", "accuracy", "fit_thresh_accuracy", "score_stdev"]

In [9]:
from utils import prettify_model_name

# Add prettified model names
results_df["pretty_name"] = results_df[model_col].apply(prettify_model_name)

# Build a label combining model name + prompting scheme + chat template
def make_row_label(row):
    name = row["pretty_name"]
    prompt_type = "Numeric" if row[numeric_prompt_col] else "Q&A"
    chat_suffix = " +chat" if row["uses_chat_template"] else ""
    return f"{name} [{prompt_type}{chat_suffix}]"

results_df["row_label"] = results_df.apply(make_row_label, axis=1)

In [ ]:
# Build the display table matching the paper format:
# One row per model+chat_template combo, with numeric and Q&A columns side by side

cols_to_keep = ["ece", "brier_score_loss", "roc_auc", "accuracy"]
col_rename = {"ece": "ECE", "brier_score_loss": "Brier score", "roc_auc": "AUC", "accuracy": "Acc."}

# Split into numeric and Q&A subsets
numeric_df = results_df[results_df[numeric_prompt_col] == True].copy()
qa_df = results_df[results_df[numeric_prompt_col] == False].copy()

# Create a unique row key: model name + chat template
def row_key(row):
    chat_suffix = "_chat" if row["uses_chat_template"] else ""
    return row[model_col] + chat_suffix

numeric_df["row_key"] = numeric_df.apply(row_key, axis=1)
qa_df["row_key"] = qa_df.apply(row_key, axis=1)

# Build numeric table
num_table = numeric_df.set_index("row_key")[cols_to_keep].rename(columns=col_rename).round(3)

# Build Q&A table
qa_table = qa_df.set_index("row_key")[cols_to_keep].rename(columns=col_rename).round(3)

# Join with suffixes like the paper
latex_table = qa_table.join(
    num_table,
    how="outer",
    lsuffix=" (mult. choice)",
    rsuffix=" (num)",
)

# Add chat template marker and prettified names from the full df
row_info = pd.concat([numeric_df, qa_df]).drop_duplicates(subset=["row_key"])
row_info = row_info.set_index("row_key")[["pretty_name", "uses_chat_template"]]
# Keep only one entry per row_key
row_info = row_info[~row_info.index.duplicated(keep="first")]

latex_table = latex_table.join(row_info, how="left")

# Add chat template marker
latex_table["Chat"] = latex_table["uses_chat_template"].map({True: r"\checkmark", False: ""})

# Prettify model names as the "Model" index
latex_table["Model"] = [
    row["pretty_name"]
    for _, row in latex_table.iterrows()
]
latex_table = latex_table.set_index("Model", drop=True)

# Drop helper columns
latex_table = latex_table.drop(columns=["pretty_name", "uses_chat_template"])

# Fill NaNs
latex_table = latex_table.fillna("-")

# Reorder: Chat Tpl first, then mult. choice columns, then num columns
qa_cols = [c for c in latex_table.columns if "(mult. choice)" in c]
num_cols = [c for c in latex_table.columns if "(num)" in c]
latex_table = latex_table[["Chat Tpl"] + qa_cols + num_cols]

display(latex_table)

,Chat Tpl,ECE (mult. choice),Brier score (mult. choice),AUC (mult. choice),Acc. (mult. choice),ECE (num),Brier score (num),AUC (num),Acc. (num)
Model,,,,,,,,,
Llama 3 8B,,0.182,0.221,0.844,0.733,0.179,0.261,0.550,0.391
Llama 3 8B (it),\checkmark,0.190,0.206,0.838,0.706,0.147,0.190,0.812,0.738
Llama 3.1 8B,,0.234,0.228,0.848,0.638,0.138,0.252,0.496,0.369
Llama 3.1 8B (it),,0.224,0.219,0.839,0.641,0.386,0.353,0.773,0.397
Mistral 7B (it),,0.087,0.172,0.825,0.756,0.104,0.180,0.811,0.734
Mistral 7B (it),\checkmark,0.115,0.180,0.825,0.758,0.368,0.368,0.500,0.632
Mistral 7B,,0.205,0.227,0.820,0.742,0.365,0.332,0.743,0.482
Gemma 2B,,0.148,0.253,0.642,0.385,0.367,0.368,0.498,0.632


## Colored LaTeX table

In [ ]:
higher_is_better_cols = {"AUC", "Acc."}

def latex_colored_float_format(val, all_values, higher_is_better=True):
    """Map a cell's value to its colored latex code."""
    min_val, max_val = np.min(all_values), np.max(all_values)
    low_pct_val, high_pct_val = [
        min_val + (max_val - min_val) * interp_point
        for interp_point in [0.1, 0.9]
    ]

    if low_pct_val <= val <= high_pct_val:
        return f"{val:.2f}"

    if val < low_pct_val:
        color = "orange" if higher_is_better else "cyan"
        color_value = 100 * ((low_pct_val - val) / (low_pct_val - min_val))
    elif val > high_pct_val:
        color = "cyan" if higher_is_better else "orange"
        color_value = 100 * ((val - high_pct_val) / (max_val - high_pct_val))
    else:
        raise RuntimeError(f"{val}")

    color_value /= 4
    return r"\cellcolor{" + f"{color}!{color_value:.1f}" + r"} " + f"{val:.2f}"


# Apply coloring to each metric column
colored_table = latex_table.copy()

for col in colored_table.columns:
    if col == "Chat":
        continue

    # Get numeric values for this column (skip "-" entries)
    col_vals = pd.to_numeric(colored_table[col], errors="coerce")
    if col_vals.isna().all():
        continue

    new_col = []
    for i in range(len(colored_table)):
        val = col_vals.iloc[i]
        if pd.isna(val):
            new_col.append("-")
        else:
            new_col.append(
                latex_colored_float_format(
                    val=val,
                    all_values=col_vals.dropna(),
                    higher_is_better=any(h in col for h in higher_is_better_cols),
                )
            )
    colored_table[col] = new_col

# -- Build LaTeX string manually for multicolumn header --

qa_cols = [c for c in colored_table.columns if "(mult. choice)" in c]
num_cols = [c for c in colored_table.columns if "(num)" in c]
n_qa = len(qa_cols)
n_num = len(num_cols)

# Short metric names (strip the suffix)
def short_name(col):
    return col.replace(" (num)", "").replace(" (mult. choice)", "")

col_align = "l" + "c" * (1 + n_qa + n_num)  # Model + Chat + metrics

lines = []
lines.append(r"\begin{tabular}{" + col_align + "}")
lines.append(r"\toprule")

# Top header row: empty for Model & Chat, then multicolumn spans
lines.append(
    r" & "  # Model col (empty in top row)
    + r" & "  # Chat col (empty in top row)
    + r"\multicolumn{" + str(n_qa) + r"}{c}{Multiple Choice}"
    + r" & \multicolumn{" + str(n_num) + r"}{c}{Numeric}"
    + r" \\"
)

# Partial rule under the group headers
lines.append(
    r"\cmidrule(lr){" + f"{3}-{2+n_qa}" + r"}"
    + r" \cmidrule(lr){" + f"{3+n_qa}-{2+n_qa+n_num}" + r"}"
)

# Sub-header row: Model, Chat, then short metric names repeated
sub_header = "Model & Chat"
for col in qa_cols + num_cols:
    sub_header += " & " + short_name(col)
sub_header += r" \\"
lines.append(sub_header)
lines.append(r"\midrule")

# Data rows — iterate by position to handle duplicate index values
for i in range(len(colored_table)):
    idx = colored_table.index[i]
    row_vals = [idx.replace("_", r"\_"), colored_table.iloc[i]["Chat"]]
    for col in qa_cols + num_cols:
        row_vals.append(str(colored_table.iloc[i][col]))
    lines.append(" & ".join(row_vals) + r" \\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")

latex_table_str = "\n".join(lines)
print(latex_table_str)

# Save to file
tables_dir = ACS_AGG_RESULTS_PATH.parent / "tables"
tables_dir.mkdir(exist_ok=True)
out_path = tables_dir / "replication-results-table.tex"
with open(out_path, "w") as f_out:
    f_out.write(latex_table_str)
print(f"\nSaved to {out_path}")

## Prompting Details

The exact prompting text used for each model configuration.

In [ ]:
from folktexts.prompting import (
    SYSTEM_PROMPT,
    NUMERIC_SYSTEM_PROMPT,
    ACS_TASK_DESCRIPTION,
    ANTHROPIC_CHAT_PROMPT,
    NUMERIC_CHAT_PROMPT,
)

# For each unique model configuration, describe the prompting that was used
prompt_info_cols = [model_col, numeric_prompt_col, "uses_chat_template", "is_inst",
                    "config_system_prompt", "config_chat_prompt", "row_label"]
unique_configs = results_df[prompt_info_cols].drop_duplicates()

def latex_escape(text):
    """Escape special LaTeX characters."""
    replacements = {
        '\\': r'\textbackslash{}',
        '&': r'\&', '%': r'\%', '$': r'\$', '#': r'\#',
        '_': r'\_', '{': r'\{', '}': r'\}', '~': r'\textasciitilde{}',
        '^': r'\textasciicircum{}',
    }
    for char, repl in replacements.items():
        text = text.replace(char, repl)
    return text

def latex_texttt_block(text):
    """Format a block of text as \\noindent\\texttt{...}, handling newlines."""
    lines = text.strip().split("\n")
    escaped = [latex_escape(line) for line in lines]
    return r"\noindent\texttt{" + r" \\ ".join(escaped) + "}"

latex_lines = []
latex_lines.append(r"\subsection*{Prompting Details}")
latex_lines.append("")
latex_lines.append("The exact prompting configuration used for each model run.")
latex_lines.append("")

for _, row in unique_configs.iterrows():
    is_numeric = bool(row[numeric_prompt_col])
    uses_chat = bool(row["uses_chat_template"])
    custom_sys = row["config_system_prompt"]
    custom_chat = row["config_chat_prompt"]

    prompt_mode = "Numeric risk prompting" if is_numeric else "Multiple-choice Q\\&A prompting"
    template_mode = "with chat template" if uses_chat else "without chat template (raw text completion)"

    label = latex_escape(row["row_label"])

    latex_lines.append(r"\paragraph{" + label + "}")
    latex_lines.append(r"\textbf{Prompting mode:} " + prompt_mode + r" \\")
    latex_lines.append(r"\textbf{Chat template:} " + template_mode)
    latex_lines.append("")

    latex_lines.append(r"\noindent \textbf{Task description} (prepended to each sample):")
    latex_lines.append("")
    latex_lines.append(latex_texttt_block(ACS_TASK_DESCRIPTION.strip()))
    latex_lines.append("")

    if uses_chat:
        if pd.notna(custom_sys) and str(custom_sys).strip():
            sys_prompt = str(custom_sys)
        else:
            sys_prompt = NUMERIC_SYSTEM_PROMPT if is_numeric else SYSTEM_PROMPT

        if pd.notna(custom_chat) and str(custom_chat).strip():
            chat_prompt = str(custom_chat)
        else:
            chat_prompt = NUMERIC_CHAT_PROMPT if is_numeric else ANTHROPIC_CHAT_PROMPT

        latex_lines.append(r"\noindent \textbf{System prompt} (in system role):")
        latex_lines.append("")
        latex_lines.append(latex_texttt_block(sys_prompt.strip()))
        latex_lines.append("")

        latex_lines.append(r"\noindent \textbf{Assistant prefill} (in assistant role; model continues from here):")
        latex_lines.append("")
        latex_lines.append(latex_texttt_block(chat_prompt.strip()))
        latex_lines.append("")

        latex_lines.append(r"\noindent \textit{Prompt is wrapped using the model's tokenizer chat template with system/user/assistant roles.}")
        latex_lines.append("")

    else:
        if is_numeric:
            question_suffix = "Question: <question text>\nAnswer (between 0 and 1): 0."
        else:
            question_suffix = "Question: <question text>\nA. <option A>.\nB. <option B>.\nAnswer:"

        latex_lines.append(r"\noindent \textbf{No system prompt or chat template wrapping} --- raw text completion mode.")
        latex_lines.append("")
        latex_lines.append(r"\noindent The full prompt is the task description + row data + question, ending with:")
        latex_lines.append("")
        latex_lines.append(latex_texttt_block(question_suffix))
        latex_lines.append("")
        latex_lines.append(r"\noindent \textit{The model generates the next token(s) directly from this raw text. No system prompt or assistant prefill is used.}")
        latex_lines.append("")

    latex_lines.append(r"\bigskip")
    latex_lines.append("")

latex_output = "\n".join(latex_lines)
print(latex_output)

# Save to file alongside the table
out_path = ACS_AGG_RESULTS_PATH.parent / "tables" / "prompting-details.tex"
with open(out_path, "w") as f_out:
    f_out.write(latex_output)
print(f"\nSaved to {out_path}")

/home/sarahep/.conda/envs/folktexts/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

: 